# Control de Lectura 2
## Condicionamiento numérico y estabilidad

SVD directa frente a formar `X.T @ X`.

## 1. Predicción registrada

**P1.** El caso más difícil será el **B** (`1e-4`, `float32`). El valor pequeño es del orden de `separacion**2` = 1e-8 frente a un valor grande de 6. Formar `X.T @ X` eleva los números al cuadrado y duplica la pérdida de cifras: esa contribución cae bajo el epsilon de `float32` (1.19e-07) y desaparece al redondear. A tiene el mismo tipo pero un valor pequeño 10.000 veces mayor; C tiene la misma separación que B pero epsilon 2.2e-16.

## 2. Casos de prueba

In [1]:
import numpy as np

CASOS = [("A", 1e-2, np.float32), ("B", 1e-4, np.float32), ("C", 1e-4, np.float64)]


def construir(sep, tipo):
    """Matriz 3x2. Al reducir sep las columnas se vuelven casi iguales."""
    return np.array([[1.0, 1.0], [1.0, 1.0 + sep], [1.0, 1.0 - sep]], dtype=tipo)

## 3. Los dos procedimientos

Ambas secuencias se ordenan de mayor a menor para poder compararlas.

In [2]:
def secuencias(X):
    """sigma^2 por SVD directa y autovalores de X.T @ X, ambos de mayor a menor."""
    sig2 = np.sort(np.linalg.svd(X, compute_uv=False) ** 2)[::-1]
    lam = np.sort(np.linalg.eigvalsh(X.T @ X))[::-1]
    return sig2, lam


resultados = [(n, s, t, *secuencias(construir(s, t))) for n, s, t in CASOS]

for nombre, _, _, sig2, lam in resultados:
    print(f"{nombre}  sigma^2 = {sig2[0]:.10e}  {sig2[1]:.10e}")
    print(f"   lambda = {lam[0]:.10e}  {lam[1]:.10e}")

A  sigma^2 = 6.0000996590e+00  9.9998149381e-05
   lambda = 6.0001001358e+00  9.9895718449e-05
B  sigma^2 = 6.0000004768e+00  1.0003319062e-08
   lambda = 6.0000000000e+00  0.0000000000e+00
C  sigma^2 = 6.0000000100e+00  9.9999999833e-09
   lambda = 6.0000000100e+00  9.9999999392e-09


## 4. Resultados

In [3]:
ENCABEZADO = ("caso", "sep", "tipo", "menor sigma^2", "menor lambda", "igual")
ANCHOS = (6, 8, 9, 22, 22, 7)


def fila(valores):
    """Une los valores ya formateados segun ANCHOS."""
    return "".join(f"{v:>{w}}" for v, w in zip(valores, ANCHOS, strict=True))


print(fila(ENCABEZADO))
print("-" * sum(ANCHOS))
for nombre, sep, tipo, sig2, lam in resultados:
    menor_sig2, menor_lam = float(sig2[-1]), float(lam[-1])
    igual = "si" if np.isclose(menor_sig2, menor_lam, rtol=1e-6, atol=0.0) else "no"
    valores = nombre, f"{sep:g}", np.dtype(tipo).name
    print(fila(valores + (f"{menor_sig2:.10e}", f"{menor_lam:.10e}", igual)))

  caso     sep     tipo         menor sigma^2          menor lambda  igual
--------------------------------------------------------------------------
     A    0.01  float32      9.9998149381e-05      9.9895718449e-05     no
     B  0.0001  float32      1.0003319062e-08      0.0000000000e+00     no
     C  0.0001  float64      9.9999999833e-09      9.9999999392e-09     si


En **B** la vía `X.T @ X` devuelve **cero exacto**: la dirección pequeña se pierde por completo.

Sobre el valor de referencia: con `c1 = (1, 1, 1)` y `c2 = c1 + sep·(0, 1, -1)`, la matriz de Gram es `[[3, 3], [3, 3 + 2·sep²]]`, cuyo autovalor pequeño resulta ser **`sep²`** — 1e-4 en A y 1e-8 en B y C. Es el valor que reproducen ambos procedimientos cuando la precisión alcanza.

## Preguntas de interpretación

**P1 (verificación).** Confirmada. B es el único caso donde el valor pequeño se pierde del todo: `X.T @ X` devuelve cero exacto mientras la SVD directa lo conserva.

**P2. A frente a B — condicionamiento.** Solo cambia la separación: la segunda columna pasa de `(1, 1.01, 0.99)` a `(1, 1.0001, 0.9999)`, cien veces más parecida a la columna de unos. El menor valor cae con el factor 1e-4 que predice `separacion**2`, y su razón contra el valor grande pasa de 1.7e-05 a 1.7e-09. Lo que empeora es el problema, no el algoritmo: el condicionamiento es propiedad **del problema**.

**P3. B frente a C — estabilidad.** En `float32` la vía `X.T @ X` pierde la dirección pequeña: su diagonal vale 3 + 2·`separacion**2`, contribución de 7e-09 que queda bajo el epsilon (1.19e-07) y se pierde al construir la matriz, antes de `eigvalsh`. En `float64` (eps 2.2e-16) ambas vías la recuperan. Mismo problema, distinta aritmética, distinto resultado: eso es inestabilidad del **procedimiento**, y formar `X.T @ X` la causa al elevar al cuadrado el número de condición.

**P4. Afirmación del estudiante.** Se **rechaza**. Las columnas no son proporcionales para ninguna separación distinta de cero, así que la matriz tiene rango 2. La prueba está en C: misma matriz y misma separación que B, y en `float64` el mismo procedimiento devuelve 9.9999999392e-09 en lugar de cero — si el cero fuera propiedad de la matriz, aparecería también ahí. La SVD directa acertó: coincide con la referencia en `float64` y con el teórico 1e-8.

Para detectar direcciones pequeñas: SVD directa sobre `X` en `float64`, contrastando contra una tolerancia explícita del tipo `max(sigma) * max(m, n) * eps` — el criterio de `np.linalg.matrix_rank` — en vez de leer un cero como rango deficiente.

**P5. Conclusiones.**

**(a) El condicionamiento se observa en este experimento cuando** acercar las columnas hunde el valor pequeño de 9.9998149381e-05 (A) a 1.0003319062e-08 (B) con el mismo `float32`, mientras el grande sigue en 6: son los datos, no el método.

**(b) La estabilidad del procedimiento se observa cuando** B y C, el mismo problema, divergen según cómo se calcule: en `float32` la vía `X.T @ X` da cero y la SVD directa 1.0003319062e-08; en `float64` ambas dan 1e-8.